In [1]:
from datetime import datetime
from pathlib import Path

import os

from dotenv import load_dotenv
import pandas as pd
import pymysql

In [2]:
PROJECT_DIR = Path.cwd().resolve().parents[1] / 'anaconda'

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

In [3]:
INPUT_CSV_PATH = (
    PROCESSED_DIR
    / 'site_01_cleaned_cats_20260829.csv'
)

cats_df = pd.read_csv(
    INPUT_CSV_PATH
)

cats_df.head()

,name,breed,gender,age,age_month,color,feature,branch,detail_url,img_url
0,홀앙,브리티쉬 먼치킨,F,2개월령,2.0,골드바이,왕눈이에 생긴 것 마냥 성격도 순둥이!,왕십리점,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
1,쨈,브리티쉬 먼치킨,M,2개월령,2.0,크림바이,오밀조밀 공주 이목구비지만 멋쟁이 왕자다냥!,잠실점,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
2,리즈,랙돌,F,2개월령,2.0,블루바이,품에 안기는 순간 인형처럼 스르륵 녹아내리는 원조 봉제인형 냥이,잠실점,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
3,우디,브리티쉬 숏헤어,M,2개월령,2.0,블루골드,느긋한 발걸음으로 집사 뒤를 묵묵히 쫓아오는 든든한 친구,왕십리점,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...
4,통깨,노르웨이숲,F,2개월령,2.0,블루태비앤화이트,NaN,왕십리점,https://www.dalunacats.com/product/item.php?ca...,https://www.dalunacats.com/product/data/item/1...


In [4]:
print(cats_df.shape)
print(cats_df.columns.tolist())

(1461, 10)
['name', 'breed', 'gender', 'age', 'age_month', 'color', 'feature', 'branch', 'detail_url', 'img_url']


In [5]:
load_dotenv(
    PROJECT_DIR / '.env'
)

True

In [6]:
DB_CONFIG = {
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT')),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME'),
    'charset': 'utf8mb4',
}

In [7]:
connection = pymysql.connect(
    **DB_CONFIG
)

print('MySQL 연결 성공')

MySQL 연결 성공


In [8]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS cat_adoption_data (
    id BIGINT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    breed VARCHAR(100) NOT NULL,
    gender VARCHAR(10) NULL,
    age VARCHAR(50) NULL,
    age_month INT NULL,
    color VARCHAR(100) NULL,
    feature VARCHAR(255) NULL,
    branch VARCHAR(100) NULL,
    detail_url VARCHAR(500) NOT NULL,
    img_url VARCHAR(500) NOT NULL,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,

    UNIQUE KEY uk_detail_url (detail_url)
)
"""

In [9]:
connection = pymysql.connect(
    **DB_CONFIG
)

try:
    with connection.cursor() as cursor:
        cursor.execute(create_table_sql)

    connection.commit()

    print('테이블 생성 완료')

finally:
    connection.close()

테이블 생성 완료


In [10]:
insert_sql = """
INSERT INTO cat_adoption_data (
    name,
    breed,
    gender,
    age,
    age_month,
    color,
    feature,
    branch,
    detail_url,
    img_url
)
VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s
)
"""

In [11]:
insert_data = []

for row in cats_df.itertuples(index=False):
    insert_data.append(
        (
            row.name,
            row.breed,
            None if pd.isna(row.gender) else row.gender,
            None if pd.isna(row.age) else row.age,
            None if pd.isna(row.age_month) else int(row.age_month),
            None if pd.isna(row.color) else row.color,
            None if pd.isna(row.feature) else row.feature,
            None if pd.isna(row.branch) else row.branch,
            row.detail_url,
            row.img_url,
        )
    )

In [12]:
print(len(insert_data))
print(insert_data[0])

1461
('홀앙', '브리티쉬 먼치킨', 'F', '2개월령', 2, '골드바이', '왕눈이에 생긴 것 마냥 성격도 순둥이!', '왕십리점', 'https://www.dalunacats.com/product/item.php?ca_id=1010&it_id=1786261670', 'https://www.dalunacats.com/product/data/item/1786261670/1786261670_s1.jpg')


In [13]:
connection = pymysql.connect(
    **DB_CONFIG
)

try:
    with connection.cursor() as cursor:
        cursor.executemany(
            insert_sql,
            insert_data,
        )

    connection.commit()

    print(
        f'{len(insert_data)}건 INSERT 완료'
    )

except Exception as e:
    connection.rollback()
    print('INSERT 실패:', e)

finally:
    connection.close()

1461건 INSERT 완료


In [14]:
connection = pymysql.connect(
    **DB_CONFIG
)

try:
    with connection.cursor() as cursor:
        cursor.execute(
            'SELECT COUNT(*) FROM cat_adoption_data'
        )

        db_count = cursor.fetchone()[0]

    print('DataFrame 행 수 :', len(cats_df))
    print('DB 저장 행 수    :', db_count)

finally:
    connection.close()

DataFrame 행 수 : 1461
DB 저장 행 수    : 1461


In [16]:
connection = pymysql.connect(
    **DB_CONFIG
)

try:
    with connection.cursor() as cursor:
        cursor.execute("""
            SELECT
                id,
                name,
                breed,
                gender,
                age_month,
                color,
                branch
            FROM cat_adoption_data
            ORDER BY id
            LIMIT 10
        """)

        rows = cursor.fetchall()

    for row in rows:
        print(row)

finally:
    connection.close()

(1, '홀앙', '브리티쉬 먼치킨', 'F', 2, '골드바이', '왕십리점')
(2, '쨈', '브리티쉬 먼치킨', 'M', 2, '크림바이', '잠실점')
(3, '리즈', '랙돌', 'F', 2, '블루바이', '잠실점')
(4, '우디', '브리티쉬 숏헤어', 'M', 2, '블루골드', '왕십리점')
(5, '통깨', '노르웨이숲', 'F', 2, '블루태비앤화이트', '왕십리점')
(6, '참깨', '노르웨이숲', 'F', 2, '블루바이', '왕십리점')
(7, '크로플', '먼치킨 미뉴엣', 'F', 2, '블루골드', '잠실점')
(8, '해', '브리티쉬 숏헤어', 'F', 2, '골드', '잠실점')
(9, '루빈', '브리티쉬 숏헤어', 'M', 2, '블루골드', '잠실점')
(10, '다온', '샴', 'M', 2, '씰포인트', '마포점')
